# Extract Achievements Data

Extracts Achievements assets and their data (description, points) from Anno 117
into CSV and JSON files. Output goes to `results/tables/`.

Run all cells from the project root.

In [1]:
from pathlib import Path
import json
import re

import pandas as pd

from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import AssetCache

## Load assets

In [2]:
config = Config.from_json("config.json")
assets = AssetCache.load(config)
templates = assets.templates

print(f"Total assets: {len(assets.elements)}")
print(f"Total texts: {len(assets.texts.elements)}")

Total assets: 30707
Total texts: 32762


## Build the AchievementSet -> Achievement lookup

Make usage of `AchievementSet` (12 assets) that holds `AchievementSet.Achievements` list of assets, where each item references a `Achievement` asset.

In [3]:
from typing import Literal, TypedDict

from assetextractor.parsing.core.assets import Asset
from assetextractor.parsing.core.attributes import WandImageProto

## Access the assets.
achievement_set = templates["AchievementSet"].assets

class IconData(TypedDict):
    name: str | None
    # url: str | None
    image: WandImageProto | None
    path: str | None

def get_icon_data(asset: Asset) -> IconData:
    """Consolidates icon metadata, URL, image object, and source file path."""
    icon = asset.icon
    
    # 1. Get the name (stem) from the filename if it exists
    name = None
    path = None
    icon_node = asset.find("Standard.IconFilename")
    if icon_node and icon_node.value:
        name = icon_node.value.stem
        path = str(icon_node.value) # The original .dds path

    return {
        "name": name,
        # "url": icon.get_data_url() if icon else None,
        "image": icon.get_image() if icon else None,
        "path": path
    }

class Achievement(TypedDict):
    '''Achievement metadata'''
    uid: str
    name: str # Standard.Name
    title: dict[str, str] # I18N Localization
    description: dict[str, str] # I18N Localization
    icon: IconData
    difficulty: Literal['Bronze', 'Silver', 'Gold']
    points: int

class AchievementSet(TypedDict):
    '''Consolidate each set of achievements metadata'''
    uid: str
    reward: dict[str, str] # I18N Localization
    category: str
    achievements: list[Achievement]


## Initialize the achievement sets dictionary.
achievement_sets_dict: dict[str, AchievementSet] = {}

## Initialize the rows for Pandas Dataframe processing.
rows: list[dict] = []

# Mapping for points based on difficulty
DIFFICULTY_POINTS = {
    "Bronze": 15,
    "Silver": 30,
    "Gold": 50
}

## Loop over each set.
for set_item in achievement_set:
    set_guid = set_item.find("Standard.GUID").value
    set_cat = set_item.find("Standard.Name").value
    set_text_node = set_item.find("Text.OasisId")

    reward_loc = set_text_node().values if set_text_node and set_text_node() else {"english": "Unknown"}

    # Initialize the Set container
    achievement_sets_dict[set_guid] = {
        "uid": set_guid,
        "reward": reward_loc,
        "category": set_cat,
        "achievements": []
    }

    print(f"Set UID: {set_guid} - Category: {set_cat} - Reward Title: {reward_loc.get("english")}")

    set_ach_list = set_item.find("AchievementSet.Achievements") or []
    for set_ach in set_ach_list:
        ach_asset: Asset = set_ach.get("Asset").value
        if not ach_asset:
            continue

        ach_guid = ach_asset.guid

        # Extract "Achievement" node
        achievement_node = ach_asset.find("Achievement")
        name = ach_asset.find_value("Standard.Name")

        # Extract localized strings
        title_node = achievement_node.find("AchievementTitle")
        desc_node = achievement_node.find("AchievementDescription")

        title_map = title_node().values if title_node and title_node() else {"english": "No Title"}
        desc_map = desc_node().values if desc_node and desc_node() else {"english": "No Description"}

        # Determine Difficulty and calculate Points
        diff_node = achievement_node.find("AchievementDifficulty")
        # Default to 'Bronze' if the node is missing or empty
        difficulty_value: Literal['Bronze', 'Silver', 'Gold'] = diff_node.value if diff_node and diff_node.value else "Bronze"

        # Look up points based on the difficulty string
        points_value = DIFFICULTY_POINTS.get(difficulty_value, 15)

        # Icon metadata
        icon_data = get_icon_data(ach_asset)
        
        # Construct the Achievement object
        achievement_data: Achievement = {
            "uid": ach_guid,
            "name": name,
            "title": title_map,
            "description": desc_map,
            "icon": icon_data,
            "difficulty": difficulty_value,
            "points": points_value,
        }

        # Append to the list inside the current set
        achievement_sets_dict[set_guid]["achievements"].append(achievement_data)

        rows.append({
            "Set Name": set_cat,
            "Set UID": set_guid,
            "Name": name,
            "Icon": icon_data["name"],
            "Title": title_map.get("english"),
            "Description": desc_map.get("english"),
            "Difficulty": difficulty_value,
            "Points": points_value,
        })
        
        # print(f" ** Achievement: {ach_guid} | {title_map.get('english')} - {desc_map.get('english')}")
        # print(f"  * Difficulty: {difficulty_value} | Points: {points_value}")

    # print(f"---------------------------")

Set UID: 43710 - Category: Set01 (Construction) - Reward Title: Visionary Architect
Set UID: 80645 - Category: Set02 (Economy) - Reward Title: Efficient Economist
Set UID: 80674 - Category: Set03 (Logistic) - Reward Title: Farsighted Administrator
Set UID: 80675 - Category: Set04 (Diplomacy) - Reward Title: Tactful Diplomat
Set UID: 80676 - Category: Set05 (Elimination) - Reward Title: Ultimate Victor
Set UID: 80982 - Category: Set06 (Narration) - Reward Title: Keen Listener
Set UID: 80983 - Category: Set07 (Campaign) - Reward Title: Steadfast Governor
Set UID: 80984 - Category: Set08 (Tech/Religion) - Reward Title: Insightful Sage
Set UID: 80985 - Category: Set09 (Anno) - Reward Title: Ninefold Annotator
Set UID: 80986 - Category: Set10 (Latium) - Reward Title: Loyal Roman
Set UID: 80987 - Category: Set11 (Albion) - Reward Title: Indomitable Celt
Set UID: 145324 - Category: Set12 (Volcano) - Reward Title: Volcanic Prospector


## Generate Pandas Dataframe Table

In [4]:
df = pd.DataFrame(rows)

# Create a dictionary of DataFrames keyed by Set Name
dfs_by_set = {name: group for name, group in df.groupby("Set Name")}

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print(f"Extracted {len(df)} Achievement rows")

Extracted 108 Achievement rows


## Save CSV files

In [5]:
output_dir = Path("results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(output_dir / "achievements.csv", index=False)

print(f"Saved CSVs to {output_dir.resolve()}")

Saved CSVs to D:\Anno_117_Modding\asset-extractor\assetextractor\conversion\statistics\results\tables


## Display

In [6]:
from IPython.display import display, HTML

# Define your color palette
HEADER_BG = "#5F032E"
ROW_BG = "#EBD2B8"
TEXT_COLOR = "#1D000E"

for set_name, set_df in dfs_by_set.items():
    display(HTML(f"<h2>Set: {set_name}</h2>"))
    
    # 1. Clean the dataframe for display
    display_df = set_df.drop(columns=["Set Name", "Set UID"])
    
    # 2. Apply styling
    styled_df = (display_df.style
        # Set overall row background and text color
        .set_properties(**{
            'background-color': ROW_BG,
            'color': TEXT_COLOR,
            'border': f'1px solid {HEADER_BG}',
            'padding': '8px'
        })
        # Set header background and text color
        .set_table_styles([
            {
                'selector': 'th',
                'props': [
                    ('background-color', HEADER_BG),
                    ('color', 'white'),
                    ('font-weight', 'bold'),
                    ('text-align', 'center')
                ]
            }
        ])
        .hide(axis="index") # Remove the index column
    )
    
    display(styled_df)

Name,Icon,Title,Description,Difficulty,Points
Achievement_Set1_01_First_Settlement,achievement_set01_01_0,Ab Initio Anno,Start your settlement off with its first Fishery.,Bronze,15
Achievement_Set1_03_Long_Aqueduct_99_Tiles,achievement_set01_03_0,Aqueduct Vitae,Build an Aqueduct with a length of at least 99 tiles.,Bronze,15
Achievement_Set1_02_Plebeians_250,achievement_set01_02_0,Adunatio Plebis,House a total population of 400 Plebeians.,Bronze,15
Achievement_Set1_04_Marble_Streets,achievement_set01_04_0,Marmoream Relinquo,"Have exclusively Marble Roads on an island with a Status of ""Growing Town"" (Level V) or better.",Silver,30
Achievement_Set1_05_Build_500_Ornaments,achievement_set01_05_0,Haec Ornamenta Mea,Beautify an island with 50 ornaments.,Bronze,15
Achievement_Set1_06_Have_50_Blueprints,achievement_set01_06_0,Hippodamian Planner,Have 9 Building Plans at the same time.,Bronze,15
Achievement_Set1_07_Population_100000,achievement_set01_07_0,The First Mile,"Have a total population of 50,000 across all your cities.",Bronze,15
Achievement_Set1_08_Latium_Albion_PubBuildings,achievement_set01_08_0,Pro Bono Publico,Build all Public Services in Albion and Latium.,Gold,50
Achievement_Set1_09_Megalopolis_NoPlebs_NoLiberti,achievement_set01_09_0,Patrician Tastes,"Have the ""Major City"" (Level IX) City Status on an island without any Liberti or Plebeians.",Silver,30


Name,Icon,Title,Description,Difficulty,Points
Achievement_Set2_01_Wood_Chain,achievement_set02_01_0,Axe Romana,Set up your first Timber production chain.,Bronze,15
Achievement_Set2_02_First_Bankruptcy,achievement_set02_02_0,Anno Horribilis,Trigger the first bankruptcy warning.,Bronze,15
Achievement_Set2_03_Have_333_sardines,achievement_set02_03_0,Surplus Ultra,Have 333 talenta (t) of Sardines in storage.,Bronze,15
Achievement_Set2_04_Balance_10k_Denarii,achievement_set02_04_0,Pecunia Non Olet,Have a positive income balance of 250 Denarii.,Bronze,15
Achievement_Set2_05_100%_Productivity_Loungers,achievement_set02_05_0,Perfectly Balanced,Reach a productivity of 100% at a Loungers Factory.,Bronze,15
Achievement_Set2_06_Cover_70%_With_FarmFields,achievement_set02_06_0,Outstanding In The Field,Cover 70% of an island with farm fields.,Bronze,15
Achievement_Set2_07_Pause_Building,achievement_set02_07_0,Resting On Laurels,Pause a Production building.,Bronze,15
Achievement_Set2_08_House_9_Specialists,achievement_set02_08_0,Full House,Socket 9 Specialists on one island.,Silver,30
Achievement_Set2_09_Megalopolis_With_1_TradeRoute,achievement_set02_09_0,Ens Causa Sui,"Sustain a ""Major City"" (Level IX) for 17 minutes on an island supported by a single trade route.",Gold,50


Name,Icon,Title,Description,Difficulty,Points
Achievement_Set3_01_Second_Island,achievement_set03_01_0,Terra Nova,Settle on a second island.,Bronze,15
Achievement_Set3_02_Destroy_117_Ships,achievement_set03_02_0,Oceanic Battlefield,Destroy 117 enemy ships.,Silver,30
Achievement_Set3_03_Trade_Route_9_Stations,achievement_set03_03_0,Travelling Sails Problem,Set up a Trade Route with 9 stops.,Bronze,15
Achievement_Set3_04_Quinquireme_With_Max_Military_Modules,achievement_set03_04_0,Ultima Ratio,Build a Quinquireme equipped with the maximum number of military modules.,Bronze,15
Achievement_Set3_05_Entire_Inventory_Bought,achievement_set03_05_0,Caveat Venditor,Buy all the goods stocked by a Trader at once.,Bronze,15
Achievement_Set3_06_Spend_1701_Coins_Reroll,achievement_set03_06_0,Alea Iacta Est,"Spend at least 170,100 Denarii re-rolling items for sale.",Silver,30
Achievement_Set3_07_Throw_1602_goods_overboard,achievement_set03_07_0,Neptune's Bounty,Throw 1602 talentum(t) of goods overboard.,Bronze,15
Achievement_Set3_08_Sail_2205_Nautic_Miles,achievement_set03_08_0,From Sea To Sea,"Sail 2,205 Nautical Miles.",Bronze,15
Achievement_Set3_09_Socket_27_Captains,achievement_set03_09_0,Industry of Captains,Have a Captain equipped on 27 ships.,Gold,50


Name,Icon,Title,Description,Difficulty,Points
Achievement_Set4_01_Setup_Trade_Contract,achievement_set4_01_0,Trade Union,Establish a Trade Treaty.,Bronze,15
Achievement_Set4_02_Destroy_99_Walls,achievement_set4_02_0,Quod Erat Destructum,Destroy 117 Wall segments.,Bronze,15
Achievement_Set4_03_Alliance_With_Pirates,achievement_set4_03_0,Bad Company,Ally with Voada or Syracus.,Silver,30
Achievement_Set4_04_Conquer_Villa,achievement_set4_04_0,Delenda Est,Conquer an opponent's Villa.,Bronze,15
Achievement_Set4_05_Subjugate_Rival,achievement_set03_05_0,Ex Amicitia Pax,Appoint an Allied Rival as a Specialist.,Bronze,15
Achievement_Set4_06_Gain_Tribute,achievement_set4_06_0,Virtue's Reward,Receive Tribute from a Rival.,Bronze,15
Achievement_Set4_07_Gift_2070,achievement_set4_07_0,Ex Gratia,"Gift other Parties 207,000 Denarii in total.",Bronze,15
Achievement_Set4_08_Megalopolis_With_High_Military_Maintenance,achievement_set4_08_0,Para Bellum,"Have a ""Major City"" (Level IX) with 20% of its Income dedicated to maintenance of Military buildings.",Gold,50
Achievement_Set4_09_War_with_3_rivals_then_peace,achievement_set4_09_0,Si Vis Pacem,Broker a peace after waging war with three separate Rivals.,Silver,30


Name,Icon,Title,Description,Difficulty,Points
Achievement_Set5_01_Eliminate_Dorian,achievement_set05_01_0,A Brother's Betrayal,Eliminate Dorian.,Bronze,15
Achievement_Set5_02_Eliminate_Livia,achievement_set05_02_0,Break the Cocoon,Eliminate Licia.,Bronze,15
Achievement_Set5_03_Eliminate_Tarragon,achievement_set05_03_0,Odyssey's End,Eliminate Tarragon.,Bronze,15
Achievement_Set5_04_Eliminate_Arthr,achievement_set05_04_0,The Impossible Dream,Eliminate Athr Iorgwyn.,Bronze,15
Achievement_Set5_05_Eliminate_Zarai,achievement_set05_05_0,Starfall,Eliminate Zara Nitu.,Bronze,15
Achievement_Set5_06_Eliminate_Concordia,achievement_set05_06_0,Snuffing The Flame,Eliminate Concordia.,Silver,30
Achievement_Set5_07_Eliminate_Neferneru,achievement_set05_07_0,Heavier Than A Feather,Eliminate Neferneru.,Silver,30
Achievement_Set5_08_Pro_Consul,achievement_set05_08_0,Quid Pro Quo,Attain the rank of Proconsul.,Gold,50
Achievement_Set5_09_Consul,achievement_set05_09_0,Open for Consultation,Attain the rank of Consul.,Gold,50


...

Name,Icon,Title,Description,Difficulty,Points
Achievement_Set8_01_Have_1404_Knowledge,achievement_set08_01_0,Discendo Discimus,"Have a combined total production of 1,404 Knowledge.",Bronze,15
Achievement_Set8_02_Fulfill_9_Inspirations,achievement_set08_02_0,Scientia Potestas Est,Fulfill 9 Research Inspirations.,Bronze,15
Achievement_Set8_03_Fulfill_PastAnno_Inspirations,achievement_set08_03_0,Nonanumerologist,Fulfill all Inspirations that hearken to Anno's past or future.,Bronze,15
Achievement_Set8_04_Unlock HallofFame_Tech,achievement_set08_04_0,Orbis Non Sufficit,Acquire a Discovery from the Hall of Fame.,Bronze,15
Achievement_Set8_05_Research_Again_AndAgain,achievement_set08_05_0,Ad Infinitum,Research a repeatable Discovery again... And again.,Silver,30
Achievement_Set8_06_Reach_1602_Prestige,achievement_set08_06_0,Pro Gloria,Accumulate a total value of 1602 Prestige.,Bronze,15
Achievement_Set8_07_Have_6_Wonders_Unlocked,achievement_set08_07_0,Bona Fides,Have the Veneration Effects of 6 Patron Deities active at once.,Gold,50
Achievement_Set8_08_Highest_Religious_Milestone,achievement_set08_08_0,Devoted to Devotion,Reach the fifteenth milestone of Devotion on an island.,Bronze,15
Achievement_Set8_09_All_Inspiration_1_Category,achievement_set08_09_0,Polygnostic,Unlock all Inspirations in one Research Category.,Silver,30


Name,Icon,Title,Description,Difficulty,Points
Achievement_Set9_01_Fulfill_90_Contracts,achievement_set9_01_fulfill_90_contracts_0,Contractually Obliged,Fulfill 90 Contracts.,Silver,30
Achievement_Set9_02_Make_Break_Suggestion_9_Times,achievement_set9_02_0,One... More... Chain...,Receive the suggestion to take a break 9 times.,Bronze,15
Achievement_Set9_03_nine_Techs_Queued,achievement_set9_03_0,Apodixic Appointer,Have 9 Discoveries in the Research queue.,Bronze,15
Achievement_Set9_04_Change_Skin_90_Buildings,achievement_set9_04_0,De Novo,Change a building's Skin with the Cosmetic Tool 9 times.,Bronze,15
Achievement_Set9_05_Read_9_Inforcorners,achievement_set9_05_0,Nota Bene,Open 9 Infocorners.,Bronze,15
Achievement_Set9_06_Group_9_Ships_To_Fleet,achievement_set9_06_0,Novarchus,Group 9 ships into a fleet.,Bronze,15
Achievement_Set9_07_Patron_Gods_On9_Islands,achievement_set9_07_0,Monopantheist,Dedicate 9 Islands to the same Patron God.,Bronze,15
Achievement_Set9_08_Have_9_Trade_Routes,achievement_set9_08_0,Trails in the Sea,Set up 9 Trade Routes between different islands.,Bronze,15
Achievement_Set9_09_Run_9_Colosseum_Events,achievement_set9_09_0,Morituri Te Salutant,Host 9 Events in the Amphitheater.,Gold,50


Name,Icon,Title,Description,Difficulty,Points
Achievement_Set10_01_Specialist_In_Villa,achievement_set10_01_0,Experto Crede,Have a Specialist in a Villa.,Bronze,15
Achievement_Set10_02_Latium_Exclusive_Fashion,achievement_set10_02_0,Vestis Civitatem Facit,Build every Fashion production chain in Latium.,Bronze,15
Achievement_Set10_03_Have_3_Mini_Institutions,achievement_set10_03_0,City Watched,Place all 3 Amenities in Latium.,Bronze,15
Achievement_Set10_04_Cernunnos_in_Latium,achievement_set10_04_0,Interpretatio Celtica,Appoint Cernunnos as a Patron God of an island in Latium.,Silver,30
Achievement_Set10_05_Resolve_10_Major_Incidents,achievement_set10_05_0,Semper Vigilo,Resolve 10 Major Incidents.,Bronze,15
Achievement_Set10_06_Socket_2_Specialists_Same_Villa,achievement_set10_06_0,Reunionatrix,Socket Uiscarix and Obairrix in the same Villa.,Bronze,15
Achievement_Set10_07_Patricians_Oyster_Exclusive,achievement_set10_07_0,Clammed Up,Feed your Patricians nothing but Oysters & Caviar for a total of 117 minutes.,Silver,30
Achievement_Set10_08_Have_2700_Extra_Liberti_Workforce,achievement_set10_08_0,Libertine Liberty,Have at least 2700 excess Workforce.,Bronze,15
Achievement_Set10_09_Highest_Devotion_Patrons,achievement_set10_09_0,Holier than Thou,Have the highest Devotion to your Exalted Patron God of all Governors in Latium.,Silver,30


Name,Icon,Title,Description,Difficulty,Points
Achievement_Set11_01_Settle_in_Albion,achievement_set11_01_0,Past The White Cliffs,Settle your first island in Albion.,Bronze,15
Achievement_Set11_02_Both_Celtic_Gods,achievement_set11_02_0,Tuatha Deorum,Appoint two Celtic deities as Patron Gods on your islands.,Bronze,15
Achievement_Set11_03_Aldermen_Wine,achievement_set11_03_0,In Vino Veritas,Provide Aldermen with Wine while halting their Beer supply.,Silver,30
Achievement_Set11_04_Fully_Discover_Albion,achievement_set11_04_0,Hic Svnt Dracones,Discover all of Albion.,Bronze,15
Achievement_Set11_05_Drink_With_Celts,achievement_set11_05_0,The Last Drop,Match the Celts in a drinking contest.,Bronze,15
Achievement_Set11_06_20_Interregional_TradeRoutes,achievement_set11_06_0,Cura Annonae,Connect Albion and Latium with a minimum of 3 Trade Routes.,Bronze,15
Achievement_Set11_07_drain_the_swamp_1_island,achievement_set11_07_0,Terra Firma,Remove all the marsh tiles from one of your islands.,Bronze,15
Achievement_Set11_08_2205_Celts_No_Romans,achievement_set11_08_0,Romani Ite Domum,House 2205 Celtic residents on an island without Romanising a single one.,Silver,30
Achievement_Set11_09_1000_FireSafety_Health_Happiness,achievement_set11_09_0,Shield of Albion,"Surpass 500 Health, Fire Safety and Happiness in Albion.",Gold,50


Name,Icon,Title,Description,Difficulty,Points
Achievement_Set12_01_Claim_Volcano_Island,achievement_set12_01_0,Hot Spot,"Settle the volcanic island and have it reach City Status 2: ""Small Vicus"".",Bronze,15
Achievement_Set12_02_Volcanic_Winter_Phase3,achievement_set12_02_0,Is that... snow?,Experience Volcanic Winter.,Bronze,15
Achievement_Set12_03_Finish_Volcano_Storyline,achievement_set12_03_0,Of Rage and Redemption,Finish Caecilia's storyline.,Bronze,15
Achievement_Set12_04_Coal_Mines,achievement_set12_04_0,Save the Trees,Have at least 3 coal mines and no charcoal burner on a single island.,Bronze,15
Achievement_Set12_05_Figurine_Collection,achievement_set12_05_0,Collector Craze,Produce statuettes and board games at a rate of 200t/hr.,Bronze,15
Achievement_Set12_06_Research_DLC1_Tech,achievement_set12_06_0,Natural Philosophy,Unlock all volcanic discoveries.,Bronze,15
Achievement_Set12_07_Buy_Caecilia_Specialists,achievement_set12_07_0,Sybilline Studies,Hire ten specialists at Caecilia's harbour.,Silver,30
Achievement_Set12_08_Dominant_Vulcan_Patron,achievement_set12_08_0,Burning Devotion,Establish Vulcan as globally dominant patron and as dedicated patron on 3 Islands.,Silver,30
Achievement_Set12_09_50k_Population_Volcano_Island,achievement_set12_09_0,Pompeii,Reach a population of 50.000 on the volcanic island.,Silver,30


## Export JSON

Nested dict per AchievementSet per Achievement, with the first level key being the set GUID and the second level for each achievement inside `achivements` dictionary being the achievement uid.

In [7]:
def clean_icon_path(raw_path: str | None) -> str | None:
    """Removes everything before and including '.cache' and strips extension."""
    if not raw_path:
        return None
    
    # Use partition to split at '.cache'
    # .partition returns (before, separator, after)
    _, sep, after = raw_path.partition(".cache")
    
    if sep:
        # Remove the file extension (e.g., .dds)
        return str(Path(after).with_suffix(""))
    
    return raw_path

def export_nested_achievements(sets_dict: dict[str, AchievementSet], output_path):
    final_json = {}

    for set_uid, set_data in sets_dict.items():
        # Level 1: Keyed by SetUID
        final_json[str(set_uid)] = {
            "category": set_data["category"],
            "reward": set_data["reward"],  # This is already your I18N dict
            "achievements": {}
        }

        # Level 2: Keyed by AchievementUID inside "achievements"
        for ach in set_data["achievements"]:
            ach_uid = str(ach["uid"])

            cleaned_path = clean_icon_path(ach["icon"]["path"])
            
            final_json[str(set_uid)]["achievements"][ach_uid] = {
                "title": ach["title"],             # Full I18N dict
                "description": ach["description"], # Full I18N dict
                "points": int(ach["points"]),      # Cast to standard int
                "difficulty": ach["difficulty"],
                "image_url": cleaned_path   # Using the path from icon data
            }

    # Write with ensure_ascii=False to keep localizations readable
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_json, f, indent=4)

# Execution
target_json = output_dir / "achievements.json"
export_nested_achievements(achievement_sets_dict, target_json)

print(f"Successfully exported nested JSON to {target_json}")

Successfully exported nested JSON to results\tables\achievements.json


## Image Export

Convert every achievement image from .DDS to .webp using wand + magick

In [10]:
from wand.image import Image
import os

# 1. Flatten all icons from all sets into a single list
# We use a set for 'seen_paths' to ensure icons_data only contains unique entries
icons_data: dict[str, IconData] = {}
seen_paths = set()

for set_data in achievement_sets_dict.values():
    for ach in set_data["achievements"]:
        icon = ach["icon"]
        path = icon.get("path")
        
        if path and path not in seen_paths:
            icons_data[ach["uid"]] = ach["icon"]
            seen_paths.add(path)

print(f"Extracted {len(icons_data)} unique icons.")

## Start the conversion and export logic.
print("Started Image Export")

for guid, data in icons_data.items():
    img = data["image"]
    original_path = data["path"]
    
    # Skip if there is no image or path associated with this GUID
    if not img or not original_path:
        continue
    
    # Construct output path (swap .dds for .webp)
    output_path = os.path.splitext(original_path)[0] + ".webp"
    
    # print(f"Exporting Achievement Image - GUID: {guid}")
    # print(f"  Source: {os.path.basename(original_path)}")
    # print(f"  Target: {output_path}")
    
    try:
        # We use the existing WandImageProto object
        # Ensure format is set to webp for the encoder
        img.format = 'webp'
        img.save(filename=output_path)
    except Exception as e:
        print(f"  [ERROR] Failed to export {guid}: {e}")
    
print("---")
print("Finished Exporting")

Extracted 107 unique icons.
Started Image Export
---
Finished Exporting
